# Claude Code와 오픈소스 코딩 하네스 Pi를 시스템 프롬프트 크기·확장 방식·비용 구조로 비교하고, Pi를 설치해 첫 작업을 돌려 보기

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['ANTHROPIC_API_KEY']:
    os.environ[name] = userdata.get(name)

### Pi 설치와 버전 확인

In [ ]:
%%bash
node -v
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
pi --version

### Pi 설치와 API 키로 실행

In [ ]:
%%bash
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
pi --version
export ANTHROPIC_API_KEY=YOUR_API_KEY
cd demo_pi
pi -p "test_calc.py 테스트가 실패해. calc.py를 고쳐서 pytest를 통과시켜줘." --model anthropic/claude-haiku-5-5

### 예제 만들기

In [ ]:
%%bash
mkdir -p demo_pi && cd demo_pi
cat > calc.py <<'E'
def average(nums):
    """Return the arithmetic mean of a list of numbers."""
    return sum(nums) / (len(nums) - 1)
E
cat > test_calc.py <<'E'
from calc import average

def test_average():
    assert average([2, 4, 6]) == 4
E
git init -q . && git add . && git -c user.email=a@b -c user.name=x commit -qm init
python -m pytest -q 2>&1 | tail -5

### 수정 결과 확인

In [ ]:
%%bash
cd demo_pi && git diff && python -m pytest -q | tail -3

### 세션 파일에서 토큰 읽기

In [ ]:
%%bash
f=$(ls ~/.pi/agent/sessions/*/*.jsonl); python3 - "$f" <<'E'
import json,sys
for l in open(sys.argv[1]):
    d=json.loads(l); m=d.get('message',{})
    if m.get('role')=='assistant': print(m.get('model'),json.dumps(m.get('usage')))
E

### Claude Code로 같은 작업 (비 root 사용자)

In [ ]:
%%bash
npm i -g @anthropic-ai/claude-code
useradd -m cc; cp -r demo_pi /tmp/demo_cc 2>/dev/null; chown -R cc /tmp/demo_cc
# demo_pi 는 수정된 상태이므로 git checkout 으로 되돌려 같은 시작 상태로 맞춘다
cd /tmp/demo_cc && git checkout -q calc.py && chown -R cc .
su cc -c "cd /tmp/demo_cc && git config --global --add safe.directory /tmp/demo_cc && export ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY && claude -p 'test_calc.py 테스트가 실패해. calc.py를 고쳐서 pytest를 통과시켜줘.' --model claude-haiku-5-5 --dangerously-skip-permissions --output-format json" | python3 -c "import json,sys;d=json.load(sys.stdin);print(d['num_turns'],d['total_cost_usd']);print(d['usage'])"